 Three specifications per model:
   1. Spec A — no school variables (baseline)
   2. Spec B — binary treatment (near_tier1_1km)
   3. Spec C — continuous treatment (nearest_tier1_primary_school_dist_m)

Models:
   1. OLS       — naive benchmark, interpretable coefficients
   2. Ridge     — handles multicollinearity, no feature selection

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings("ignore")

df_ols = pd.read_csv("../cleaned_datasets/df_ols.csv")
target       = "log_resale_price"
mean_price   = np.exp(df_ols[target].mean())
y            = df_ols[target]

# 1. BINARISE SCHOOL VARIABLES
# For generic schools: if count >= 1, then near = 1
df_ols["near_schools_1km"] = (df_ols["num_schools_1km"] >= 1).astype(int)
df_ols["near_schools_2km"] = (df_ols["num_schools_2km"] >= 1).astype(int)
df_ols["near_tier1_2km"]   = (df_ols["nearest_tier1_primary_school_dist_m"] <= 2000).astype(int)
df_ols["near_normalschools_1km"] = (df_ols["nearest_normal_primary_school_dist_m"] <= 1000).astype(int)
df_ols["near_normalschools_2km"] = (df_ols["nearest_normal_primary_school_dist_m"] <= 2000).astype(int)

# 2. DEFINE FEATURES
base_controls = [
    "floor_area_sqm", "num_unique_mrt_1km", "walking_dist_mrt_m",
    "walking_dist_busstop_m", "walking_dist_hawker_m", "walking_dist_mall_m",
    "dist_cbd_m", "remaining_lease_numeric",
    # Towns
    "town_BEDOK", "town_BISHAN", "town_BUKIT BATOK", "town_BUKIT MERAH", 
    "town_BUKIT PANJANG", "town_BUKIT TIMAH", "town_CENTRAL AREA", 
    "town_CHOA CHU KANG", "town_CLEMENTI", "town_GEYLANG", "town_HOUGANG", 
    "town_JURONG EAST", "town_JURONG WEST", "town_KALLANG/WHAMPOA", 
    "town_MARINE PARADE", "town_PASIR RIS", "town_PUNGGOL", "town_QUEENSTOWN", 
    "town_SEMBAWANG", "town_SENGKANG", "town_SERANGOON", "town_TAMPINES", 
    "town_TOA PAYOH", "town_WOODLANDS", "town_YISHUN",
    # Flat Types
    "flat_type_2 ROOM", "flat_type_3 ROOM", "flat_type_4 ROOM", 
    "flat_type_5 ROOM", "flat_type_EXECUTIVE",
    # Storey Ranges
    "storey_range_04 TO 06", "storey_range_07 TO 09", "storey_range_10 TO 12",
    "storey_range_13 TO 15", "storey_range_16 TO 18", "storey_range_19 TO 21",
    "storey_range_22 TO 24", "storey_range_25 TO 27", "storey_range_28 TO 30",
    "storey_range_31 TO 33", "storey_range_34 TO 36", "storey_range_37 TO 39",
    "storey_range_40 TO 42", "storey_range_43 TO 45", "storey_range_46 TO 48",
    "storey_range_49 TO 51",
    # Flat Models
    "flat_model_3Gen", "flat_model_Adjoined flat", "flat_model_Apartment",
    "flat_model_DBSS", "flat_model_Improved", "flat_model_Improved-Maisonette",
    "flat_model_Maisonette", "flat_model_Model A", "flat_model_Model A-Maisonette",
    "flat_model_Model A2", "flat_model_New Generation", "flat_model_Premium Apartment",
    "flat_model_Premium Apartment Loft", "flat_model_Premium Maisonette",
    "flat_model_Simplified", "flat_model_Standard", "flat_model_Terrace",
    "flat_model_Type S1", "flat_model_Type S2"
]



In [ ]:
#COMPARING SCHOOLS VS GOOD
# MODEL SPECS
MAIN_MODEL_VAR = {
    "Model 1 (1km)": ["near_schools_1km", "near_tier1_1km"],
    "Model 2 (2km)": ["near_schools_2km", "near_tier1_2km"],
    "Model 3 (only near school 1km)": ["near_schools_1km"],
    "Model 4 (only near school 2km)": ["near_schools_2km"]
}


# OLS
results = {}

for model_label, school_vars in MAIN_MODEL_VAR.items():
    feature_cols = base_controls + school_vars
    X = sm.add_constant(df_ols[feature_cols])
    
    mod = sm.OLS(y, X).fit(cov_type="HC3")
    
    for var in school_vars:
        beta = mod.params[var]
        pval = mod.pvalues[var]
        results[(model_label, var)] = {
            "beta": beta,
            "se": mod.bse[var],
            "p": pval,
            "sig": ("***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else "n.s."),
            "pct": (np.exp(beta) - 1) * 100,
            "adj_r2": mod.rsquared_adj,
            "n": int(mod.nobs)
        }

# SUMMARY
print(f"{'='*100}")
print(f"{'OLS RESULTS: HEDONIC PRICE PREMIUMS':^100}")
print(f"{'='*100}")
header = f"{'Model Spec':<20} {'Variable':<20} {'Beta':>10} {'p-val':>8} {'Sig':>5} {'% Effect':>12} {'Adj. R2':>10}"
print(header)
print("-" * 100)

for (model_label, var), r in results.items():
    print(f"{model_label:<20} {var:<20} {r['beta']:>10.4f} {r['p']:>8.4f} {r['sig']:>5} {r['pct']:>+11.2f}% {r['adj_r2']:>10.4f}")

print("-" * 100)

                                OLS RESULTS: HEDONIC PRICE PREMIUMS                                 
Model Spec           Variable                   Beta    p-val   Sig     % Effect    Adj. R2
----------------------------------------------------------------------------------------------------
Model 1 (1km)        near_schools_1km        -0.0532   0.0000   ***       -5.18%     0.9265
Model 1 (1km)        near_tier1_1km           0.0062   0.0000   ***       +0.62%     0.9265
Model 2 (2km)        near_schools_2km        -0.6430   0.0000   ***      -47.43%     0.9268
Model 2 (2km)        near_tier1_2km           0.0195   0.0000   ***       +1.97%     0.9268
Model 3 (only near school 1km) near_schools_1km        -0.0516   0.0000   ***       -5.03%     0.9265
Model 4 (only near school 2km) near_schools_2km        -0.6390   0.0000   ***      -47.22%     0.9266
----------------------------------------------------------------------------------------------------


In [ ]:
#COMPARING NORMALSCHOOLS VS GOOD
# MODEL SPECS
MAIN_MODEL_VAR = {
    "Model 1 (1km)": ["near_normalschools_1km", "near_tier1_1km"],
    "Model 2 (2km)": ["near_normalschools_2km", "near_tier1_2km"],
    "Model 3 (only near school 1km)": ["near_normalschools_1km"],
    "Model 4 (only near school 2km)": ["near_normalschools_2km"]
}

# OLS
results = {}

for model_label, school_vars in MAIN_MODEL_VAR.items():
    feature_cols = base_controls + school_vars
    X = sm.add_constant(df_ols[feature_cols])
    
    mod = sm.OLS(y, X).fit(cov_type="HC3")
    
    for var in school_vars:
        beta = mod.params[var]
        pval = mod.pvalues[var]
        results[(model_label, var)] = {
            "beta": beta,
            "se": mod.bse[var],
            "p": pval,
            "sig": ("***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else "n.s."),
            "pct": (np.exp(beta) - 1) * 100,
            "adj_r2": mod.rsquared_adj,
            "n": int(mod.nobs)
        }

# SUMMARY
print(f"{'='*100}")
print(f"{'OLS RESULTS: HEDONIC PRICE PREMIUMS':^100}")
print(f"{'='*100}")
header = f"{'Model Spec':<20} {'Variable':<20} {'Beta':>10} {'p-val':>8} {'Sig':>5} {'% Effect':>12} {'Adj. R2':>10}"
print(header)
print("-" * 100)

for (model_label, var), r in results.items():
    print(f"{model_label:<20} {var:<20} {r['beta']:>10.4f} {r['p']:>8.4f} {r['sig']:>5} {r['pct']:>+11.2f}% {r['adj_r2']:>10.4f}")

print("-" * 100)

                                OLS RESULTS: HEDONIC PRICE PREMIUMS                                 
Model Spec           Variable                   Beta    p-val   Sig     % Effect    Adj. R2
----------------------------------------------------------------------------------------------------
Model 1 (1km)        near_normalschools_1km    -0.0258   0.0000   ***       -2.55%     0.9261
Model 1 (1km)        near_tier1_1km           0.0012   0.0860     *       +0.12%     0.9261
Model 2 (2km)        near_normalschools_2km    -0.1123   0.0000   ***      -10.62%     0.9261
Model 2 (2km)        near_tier1_2km           0.0192   0.0000   ***       +1.93%     0.9261
Model 3 (only near school 1km) near_normalschools_1km    -0.0259   0.0000   ***       -2.56%     0.9261
Model 4 (only near school 2km) near_normalschools_2km    -0.1097   0.0000   ***      -10.39%     0.9259
----------------------------------------------------------------------------------------------------
